In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

In [ ]:

env = pd.read_excel(
    "data.xlsx",
    sheet_name="Environment_Summary",
    header=1 # # lay ten cot
)

result = env[["Condition", "Average Error (%)"]].dropna()

print(result)

plt.figure(figsize=(8, 5))
plt.bar(result["Condition"], result["Average Error (%)"])

plt.xlabel("Condition")
plt.ylabel("Average Error (%)")
plt.title("Average Error under Different Environmental Conditions")
plt.xticks(rotation=15)

plt.show()

### Conclusion

The average measurement error under all tested environmental conditions was between about 1.65% and 2.45%.

The difference between the conditions was small, so airflow and background noise had only a small effect on the distance measurement results.

In [ ]:
wire = pd.read_excel(
    "data.xlsx",
    sheet_name="Wire_Length",
    header=1
)

wire = wire[wire["Trial"].notna()].copy() # chir lay dong cos trial

result = pd.DataFrame({
    "Wire": ["Short wire (20 cm)", "Long wire (2 m)"],
    "Average Error (%)": [
        wire["Error_Short_pct"].mean() * 100,
        wire["Error_Long_pct"].mean() * 100
    ]
})

print(result.round(3)) # 3 cs tp

plt.figure(figsize=(7, 5))
plt.bar(
    result["Wire"],
    result["Average Error (%)"]
)

plt.ylabel("Average Error (%)")
plt.title("Effect of Wire Length on Distance Measurement")

plt.show()

### Conclusion

The average error was 0.813% for the 20 cm wire and 0.850% for the 2 m wire.

The difference was very small. Therefore, increasing the wire length from 20 cm to 2 m had little effect on the distance measurement results.

In [ ]:

ori = pd.read_excel("data.xlsx", sheet_name="Orientation", header=1)

ori["Actual_cm"] = pd.to_numeric(ori["Actual_cm"], errors="coerce") #nan
ori["Measured_cm"] = pd.to_numeric(ori["Measured_cm"], errors="coerce")
ori = ori.dropna(subset=["Actual_cm", "Measured_cm"])

ori["Error_pct"] = (
    (ori["Measured_cm"] - ori["Actual_cm"]).abs() / ori["Actual_cm"] * 100
)

result = (
    ori.groupby("Actual_cm")["Error_pct"]
    .mean()
    .reset_index(name="Average Error (%)")
)

print(result.round(3))
print(
    "\nOverall Average Error:",
    round(ori["Error_pct"].mean(), 3),
    "%",
)

plt.figure(figsize=(7, 5))
plt.bar(result["Actual_cm"].astype(str), result["Average Error (%)"])
plt.xlabel("Actual Distance (cm)")
plt.ylabel("Average Error (%)")
plt.title("Distance Measurement in Vertical Orientation")
plt.show()

### Conclusion

In the vertical setup, the measurement error was about 0.7%–1.2%.

This error range was similar to the Wire Length test (about 0.81%–0.85%)

Therefore, the vertical setup was considered suitable for measuring parcel height.

In [ ]:
cal = pd.read_excel(
    "data.xlsx",
    sheet_name="Calibration"
)


In [ ]:
for column in ["Actual_cm", "CB1_cm", "CB2_cm", "CB3_cm"]:
    cal[column] = pd.to_numeric(
        cal[column],
        errors="coerce"
    )

cal = cal.dropna(
    subset=["Actual_cm", "CB1_cm", "CB2_cm", "CB3_cm"]
)

test_distances = [11, 21, 31, 41, 51, 61, 71, 81, 91]

train_cal = cal[
    ~cal["Actual_cm"].isin(test_distances) #pd
].copy()

test_cal = cal[
    cal["Actual_cm"].isin(test_distances)
].copy()

print("Train samples:", len(train_cal))
print("Test samples:", len(test_cal))

In [ ]:
sensors = ["CB1_cm", "CB2_cm", "CB3_cm"]

mean_train = train_cal.groupby("Actual_cm")[sensors].mean().reset_index()
mean_test = test_cal.groupby("Actual_cm")[sensors].mean().reset_index()

print("Train means:")
print(mean_train.round(3))

print("\nTest means:")
print(mean_test.round(3))

In [ ]:

for sensor in sensors:
    sensor_name = sensor.replace("_cm", "")

    lower_limit = min(
        cal["Actual_cm"].min(),
        cal[sensor].min()
    ) - 5

    upper_limit = max(
        cal["Actual_cm"].max(),
        cal[sensor].max()
    ) + 5

    plt.figure(figsize=(10, 7))

    plt.scatter(
        mean_train["Actual_cm"],
        mean_train[sensor],
        s=60,
        label="Training distances"
    )

    plt.scatter(
        mean_test["Actual_cm"],
        mean_test[sensor],
        s=90,
        color="red",
        label="Testing distances"
    )

    plt.plot(
        [lower_limit, upper_limit],
        [lower_limit, upper_limit],
        linestyle="--",
        color="black",
        label="Ideal y = x"
    )

    plt.xlabel("Actual distance (cm)")
    plt.ylabel("Measured distance (cm)")
    plt.title(f"{sensor_name} - Training and Testing Data")

    plt.xlim(lower_limit, upper_limit)
    plt.ylim(lower_limit, upper_limit)

    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
models = {}
test_results = []

for sensor in sensors:
    sensor_name = sensor.replace("_cm", "")

    X_train = train_cal[[sensor]]
    y_train = train_cal["Actual_cm"]

    X_test = test_cal[[sensor]]
    y_test = test_cal["Actual_cm"]

    model = LinearRegression()
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    mae = mean_absolute_error(
        y_test,
        y_pred
    )

    rmse = np.sqrt(
        np.mean((y_test - y_pred) ** 2)
    )

    mape = mean_absolute_percentage_error(
        y_test,
        y_pred
    ) * 100

    models[sensor] = model

    test_results.append({
        "Sensor": sensor_name,
        "Coefficient": model.coef_[0],
        "Intercept": model.intercept_,
        "Test MAE (cm)": mae,
        "Test RMSE (cm)": rmse,
        "Test MAPE (%)": mape
    })

test_results = pd.DataFrame(test_results)

print(test_results.round(4))

In [ ]:
comparison_results = []

for sensor in sensors:
    sensor_name = sensor.replace("_cm", "")

    actual = test_cal["Actual_cm"]
    measured_before = test_cal[sensor]

    measured_after = models[sensor].predict(
        test_cal[[sensor]]
    )

    mae_before = mean_absolute_error(
        actual,
        measured_before
    )

    mae_after = mean_absolute_error(
        actual,
        measured_after
    )

    rmse_before = np.sqrt(
        np.mean((actual - measured_before) ** 2)
    )

    rmse_after = np.sqrt(
        np.mean((actual - measured_after) ** 2)
    )

    mape_before = mean_absolute_percentage_error(
        actual,
        measured_before
    ) * 100

    mape_after = mean_absolute_percentage_error(
        actual,
        measured_after
    ) * 100

    improvement = (
        (mae_before - mae_after)
        / mae_before * 100
    )

    comparison_results.append({
        "Sensor": sensor_name,
        "MAE Before (cm)": mae_before,
        "MAE After (cm)": mae_after,
        "RMSE Before (cm)": rmse_before,
        "RMSE After (cm)": rmse_after,
        "MAPE Before (%)": mape_before,
        "MAPE After (%)": mape_after,
        "MAE Improvement (%)": improvement
    })

comparison = pd.DataFrame(comparison_results)

print(comparison.round(4))

The results show that the slopes of all three models are close to 1, which means the relationship between the sensor measurements and the actual distance is nearly linear. The positive intercepts indicate that the sensors tend to read lower than the actual values and need an added correction. After calibration, CB1 achieved an MAE of 0.2363 cm, an RMSE of 0.2852 cm, and a MAPE of 0.5267%, making it the most accurate sensor. CB2 showed the greatest MAE improvement at 76.52%, but its error after calibration remained higher than CB1. Overall, linear regression significantly reduced the MAE, RMSE, and MAPE for all three sensors.

In [ ]:
for sensor in sensors:
    sensor_name = sensor.replace("_cm", "")

    actual = test_cal["Actual_cm"]
    before = test_cal[sensor]

    after = models[sensor].predict(
        test_cal[[sensor]]
    )

    error_by_distance = pd.DataFrame({
        "Actual_cm": actual,
        "Error Before": abs(actual - before),
        "Error After": abs(actual - after)
    })

    error_by_distance = (
        error_by_distance.groupby("Actual_cm")
        [["Error Before", "Error After"]]
        .mean()
        .reset_index()
    )

    plt.figure(figsize=(9, 5))

    plt.plot(
        error_by_distance["Actual_cm"],
        error_by_distance["Error Before"],
        marker="o",
        label="Before Linear Regression"
    )

    plt.plot(
        error_by_distance["Actual_cm"],
        error_by_distance["Error After"],
        marker="o",
        label="After Linear Regression"
    )

    plt.xlabel("Actual Distance (cm)")
    plt.ylabel("Mean Absolute Error (cm)")
    plt.title(
        f"{sensor_name} - Test Error Before and After Calibration"
    )

    plt.xticks(test_distances)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
for sensor in sensors:
    sensor_name = sensor.replace("_cm", "")
    model = models[sensor]

    coefficient = model.coef_[0]
    intercept = model.intercept_

    sign = "+" if intercept >= 0 else "-"

    print(
        f"{sensor_name}: Actual = "
        f"{coefficient:.6f} × Measured "
        f"{sign} {abs(intercept):.6f}"
    )